In [1]:
import os
import django
import sys
# Set up Django environment
sys.path.append('C:\\ROHITCHOUHAN\\PORTAL\\deidentification\\deIdentification')
os.environ.setdefault("DJANGO_SETTINGS_MODULE", "deIdentification.settings")
os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "true"
django.setup()

from nd_api.views.de_identification_task import create_deidentification_task
from nd_api.models import DbDetailsModel, TableDetailsModel, IgnoreRowsDeIdentificaiton
from worker.models import Task, Chain
from django.contrib.auth.models import User
from keycloakauth.rolemodel import RoleModel, get_default_permissions
from nd_scripts.create_account import create_account
from nd_api.views.db_views import create_stats_generation_tasks
from core.process.main import start_de_identification_for_table
from nd_api.views.de_identification_task import create_deidentification_task

def clean_db():
    RoleModel.objects.all().delete()
    User.objects.all().delete()
    DbDetailsModel.objects.all().delete()
    Chain.objects.all().delete()

In [80]:
len(batch1+batch2+batch3+batch4 + batch5 + batch6)

520

In [42]:
for t in ['edi_invoice',
 'emrreferralattachmentdocs',
 'userprofile',
 'labordersdetails',
 'masked_doc_id_mapping',
 'patientpharmacylist',
 'registry_currentmeds',
 'familydetail',
 'emrprogressnoteimages',
 'vitals_registry',
 'measure_encounterdata_54024',
 'measure_encounterdata_55019',
 'raceothers',
 'familyhxdetails',
 'icd_9',
 'measure_encounterdata_54023',
 'measure_encounterdata_550']:
    tb= TableDetailsModel.objects.get(table_name=t)
    for ts in Task.all_objects.filter(arguments__table_id=tb.id):
        ts.status = 0
        ts.soft_delete = False
        ts.save()
        # print("aa")

In [32]:
for t in Task.objects.filter(status=0):
    t.status= -2
    # import datetime
    # dt = datetime.datetime(2025, 3, 2, 22, 52, 17, 226679, tzinfo=datetime.timezone.utc)
    # new_dt = dt + datetime.timedelta(days=10)
    # t.expires_at = new_dt 
    t.save()

In [13]:
Task.objects.filter(status=0)[0].expires_at

datetime.datetime(2025, 3, 12, 22, 52, 17, 226679, tzinfo=datetime.timezone.utc)

In [26]:
large_tables = ['encounters',
 'family',
 'oldrxformularydetails',
 'hl7labnotes',
 'reconciliation',
 'rxhub_rxhistory',
 'eptstmtstemp',
 'annualnotes',
 'oldrxdetail',
 'oldrxmain',
 'labattachment_archive',
 'diagnosis',
 'review',
 'surescript_eprescription',
 'familydata',
 'cre_preview_event',
 'ecliniforms',
 'emrereferralattachmentsxsl',
 'electronichl7content_archive2',
 'electronichl7content',
 'referral',
 'patientinfo',
 'structhpi',
 'edi_inv_diagnosis',
 'docsrefdata',
 'procedurespl',
 'hpi',
 'questionimported',
 'telenc',
 'notes',
 'progressnotes',
 'structsocialhistory',
 'recelectroniclabresults',
 'tblwebmsg',
 'laborders',
 'x12ebs',
 'vitals',
 'ihe_document',
 'surgicalhistory',
 'vitalshistory',
 'questionnairedata',
 'physicalexam',
 'encounterdata',
 'hospitalization',
 'labdataex',
 'labdatadetail',
 'ihe_document_bkup',
 'treatmentnotes',
 'enc',
 'cpthcpcschanges',
 'social',
 'questaoeans',
 'immunizationvismapping',
 'emrereferralattachments']


def has_notes(table_id):
    table = TableDetailsModel.objects.get(id=table_id)
    
    pid_col = [
        col['column_name'] for col in table.table_details_for_ui['columns_details'] if col["de_identification_rule"] == "NOTES"
    ]
    return len(pid_col)>0

nts = []
for t in large_tables:
    tid = TableDetailsModel.objects.get(table_name=t)
    if not has_notes(tid.id):
        nts.append(t)
nts
        
    

['oldrxformularydetails',
 'familydata',
 'ecliniforms',
 'x12ebs',
 'cpthcpcschanges',
 'immunizationvismapping']

In [65]:
def has_notes(table_id):
    table = TableDetailsModel.objects.get(id=table_id)
    
    pid_col = [
        col['column_name'] for col in table.table_details_for_ui['columns_details'] if col["de_identification_rule"] == "NOTES"
    ]
    return len(pid_col)>0
tables = []
for tb in TableDetailsModel.objects.all():
    if tb.table_status == 2:
        continue
    if tb.rows_count < 1000000:
        if  has_notes(tb.id):
            tables.append(tb.table_name)
            Task.objects.filter(arguments__table_id=tb.id).delete()
            # for t in Task.objects.filter(arguments__table_id=tb.id):
            #     t.status = 0
            #     t.save()
tables

['emrereferralattachmentsxsl',
 'labattachment_archive',
 'CQM_lookup',
 'patients',
 'pmorders',
 'reconciliation',
 'oldrxmain_addlinfo',
 'contacts',
 'assignedpnenc',
 'recelectroniclabresults',
 'hcfa',
 'cre_preview_event',
 'surescript_eprescription',
 'electronichl7content_archive2',
 'electronichl7content',
 'billingdata',
 'patientinfo',
 'referral',
 'physicalexam',
 'docsrefdata',
 'procedurespl',
 'structdemographics',
 'labdataex',
 'questionimported',
 'laborders',
 'edi_inspayments',
 'structsocialhistory',
 'problemlist',
 'ptinstruction',
 'questionnairedata',
 'structhpi',
 'structpreventive',
 'hospitalization',
 'labdata',
 'questaoeans',
 'emrereferralattachments']

In [27]:
TableDetailsModel.objects.filter(table_status=2).count()

450

In [84]:
tables = batch1+batch2+batch3+batch4+batch5+batch6
issue = []
for tb in tables:
    try:
        tbo = TableDetailsModel.objects.get(table_name=tb)
        if tbo.rows_count > 100000:
            for t in Task.objects.filter(arguments__table_id=tbo.id, status=0):
                t.status = -2
                t.save()
    except:
        issue.append(tb)

In [71]:
# Chain.objects.all().delete()

In [86]:
TableDetailsModel.objects.filter(table_name__in=batch1+batch2+batch3+batch4+batch5+batch6, table_status=2).count()

348

In [27]:
TableDetailsModel.objects.filter(table_status=2).count()
# TableDetailsModel.objects.filter(table_status=0).count()
# TableDetailsModel.objects.filter(table_status=-1).count()

455

In [14]:
list(TableDetailsModel.objects.filter(table_status=0).values_list('table_name', flat=True))

['hl7labnotes',
 'oldrxdetail',
 'oldrxmain',
 'electronichl7content_archive2',
 'electronichl7content',
 'docsrefdata',
 'surescript_eprescription',
 'patientinfo',
 'structhpi',
 'edi_inv_diagnosis',
 'procedurespl',
 'questionimported',
 'telenc',
 'structsocialhistory',
 'labdatadetail',
 'enc',
 'cpthcpcschanges',
 'questaoeans',
 'immunizationvismapping',
 'emrereferralattachments']

In [81]:
IgnoreRowsDeIdentificaiton.objects.filter(db_name="texas_prod", table_name="tblportalcontactsdecrypted",
                          )[0].__dict__

{'_state': <django.db.models.base.ModelState at 0x1ee90c686d0>,
 'id': 3997046,
 'db_name': 'texas_prod',
 'table_name': 'tblportalcontactsdecrypted',
 'row': '{"id": 458, "contactId": 80739, "pid": 172431, "name": "Tortorice, Susan", "relation": "Friend", "address": "", "address2": "", "city": "", "state": "", "zipcode": "", "homePhone": "--", "workPhone": "214-953-6859", "delflag": 0, "contactType": 1, "timeVal": {"py/object": "datetime.datetime", "__reduce__": [{"py/type": "datetime.datetime"}, ["B94EHg00EAAAAA=="]]}, "isFamilyMember": 0, "isEmergencyContact": 0, "dob": {"py/object": "datetime.date", "__reduce__": [{"py/type": "datetime.date"}, ["B2wBAQ=="]]}, "doi": 0, "cellPhone": ""}',
 'created_at': datetime.datetime(2025, 3, 5, 7, 39, 14, 140614, tzinfo=datetime.timezone.utc),
 'updated_at': datetime.datetime(2025, 3, 5, 7, 39, 14, 140614, tzinfo=datetime.timezone.utc)}

In [26]:
batch7 = ['measure_reportdata_420', 'measure_reportdata_430', 'measure_reportdata_450', 'measure_reportdata_460', 'measure_reportdata_54023', 'measure_reportdata_54024', 'measure_reportdata_550', 'measure_reportdata_55019', 'measure_reportdata_55020', 'measure_reportdata_55021', 'measure_reportdata_560', 'measure_reportdata_59019', 'measure_reportdata_59020', 'measure_reportdata_59021', 'measure_reportdata_59022', 'measure_reportdata_59023', 'measure_reportdata_59024']
TableDetailsModel.objects.filter(table_name__in=batch7, table_status=2).count()
len(batch7)

17

In [51]:
Task.objects.filter(status=0).count()

4

In [19]:
t_auto_incr = ['encounters',
 'family',
 'oldrxformularydetails',
 'hl7labnotes',
 'reconciliation',
 'rxhub_rxhistory',
 'eptstmtstemp',
 'annualnotes',
 'oldrxdetail',
 'oldrxmain',
 'labattachment_archive',
 'diagnosis',
 'review',
 'surescript_eprescription',
 'familydata',
 'cre_preview_event',
 'ecliniforms',
 'emrereferralattachmentsxsl',
 'electronichl7content_archive2',
 'electronichl7content',
 'referral',
 'patientinfo',
 'structhpi',
 'edi_inv_diagnosis',
 'docsrefdata',
 'procedurespl',
 'hpi',
 'questionimported',
 'telenc',
 'notes',
 'progressnotes',
 'structsocialhistory',
 'recelectroniclabresults',
 'tblwebmsg',
 'laborders',
 'x12ebs',
 'vitals',
 'ihe_document',
 'surgicalhistory',
 'vitalshistory',
 'questionnairedata',
 'physicalexam',
 'encounterdata',
 'hospitalization',
 'labdataex',
 'labdatadetail',
 'ihe_document_bkup',
 'treatmentnotes',
 'enc',
 'cpthcpcschanges',
 'social',
 'questaoeans',
 'immunizationvismapping',
 'emrereferralattachments']

for tn in t_auto_incr:
    t = TableDetailsModel.objects.get(table_name=tn)
    con = t.table_details_for_ui['columns_details']
    con.append({'is_phi': False,
      'mask_value': '',
      'column_name': 'nd_auto_increament_id',
      'ignore_rows': {},
      'add_to_phi_table': False,
      'de_identification_rule': '',
      'column_name_for_phi_table': None}
              )
    t.table_details_for_ui['columns_details'] = con
    t.save()
